### QuSciTech-Labs — Navigation

[Public Labs](https://github.com/jopaneur/quscitech-labs) ·
[Full Edition Access](https://github.com/jopaneur/quscitech-labs#-full-edition-kdp) ·
[Private Repo](https://github.com/jopaneur/quscitech-labs-full) ·
[QuSciTech.com](https://www.quscitech.com) ·
[The Quantum AI Book (QAIS)](https://www.amazon.com/dp/placeholder) ·

DOI: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.17212825.svg)](https://doi.org/10.5281/zenodo.17212825)

E.2 Lab 5 — Entanglement-Assisted Tamper Check — Tampered vs Untampered Distributions

### Lab Access and Execution Guide
This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional Volume).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.  

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_2_Operations_&_Scientific_Framework_of_QAIS_Advanced_Challenge_Bloch_Trajectories_Under_Composite_Gates.ipynb)



---
**Note for Lab Participants**
Each plot generated in this notebook is automatically saved as a `.png` file under: Advanced_Labs/figures/

The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  

This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**, the images are created inside the session’s working directory at:  
  `/content/Advanced_Labs/figures/`  

- When running **locally**, they appear next to your notebook files, under the subfolder:  
  `Advanced_Labs/figures/`  

- These images are **not automatically added to your GitHub repo**. They will only appear there if you manually copy, commit, and push them.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Book Reference: Chapter 10 — Quantum Communication for Distributed AI Systems**

*Chapter 10* explores quantum communication as both an enabler of distributed intelligence and a mechanism for trust verification. It shows how entanglement allows remote systems to share correlated states that can reveal tampering or unauthorized observation. In Quantum AI Systems (QAIS), this capability forms the foundation for self-verifying communication—where coherence itself becomes proof of integrity.

The chapter explains that when entangled qubits are transmitted through a channel, any interference or eavesdropping disrupts the expected measurement correlations. Such disturbances can be detected statistically, enabling physics-based authentication rather than relying on classical cryptography alone.

**Beginner Lab 10 — Entanglement-Assisted Tamper Check: Tampered vs Untampered Distributions**

Beginner Lab 10 demonstrates this principle through an entanglement-assisted tamper check. Learners prepare a Bell pair and measure correlated outcomes under both normal and disturbed conditions. In the untampered run, only perfectly correlated results (00 or 11) appear. When interference is simulated, cross-outcomes (01 and 10) emerge, signaling a breach of coherence. The experiment illustrates that QAIS communication protocols do not merely transmit information—they continually test channel integrity through entanglement correlations.

*Goal:* Use entangled qubits to detect tampering in a quantum communication channel. Learners create a Bell state, transmit one qubit through a simulated channel, and compare measurement results before and after a deliberate disturbance.

**Expected Outcome:**

* Untampered runs produce only the correlated outcomes 00 and 11.

* Tampered runs introduce cross-outcomes 01 and 10, revealing loss of entanglement fidelity and confirming channel disturbance.

This lab reinforces Chapter 10’s theme that trust in QAIS arises from physics, not protocol. Entanglement makes tampering visible, transforming communication into a continuous integrity check—an essential capability for secure, distributed AI networks.
Cross-reference: Appendix E.1, Figure E.1.10 — Tampered vs Untampered Distributions.

In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Beginner_Labs/figures",
                  fig=None, ax=None):
    """Save the current/explicit figure with a prefixed label and consistent path."""
    os.makedirs(subdir, exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))
    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print("Saved", outpath)


In [ ]:
# === Environment Setup ===
import sys, subprocess, pkgutil
def ensure(pkg):
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p in ['qiskit','qiskit-aer','matplotlib','numpy','scikit-learn']:
    ensure(p)
import qiskit, numpy as np, matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print('Qiskit:', qiskit.__version__)


---

**Lab 10: Entanglement-Assisted Tamper Check — Tampered vs Untampered Distributions**

**Using Entanglement for Integrity Verification**

This lab demonstrates how entanglement can be used to detect tampering in transmitted qubits. Alice and Bob share entangled pairs; if the transmission channel is disturbed, the correlation pattern shifts. The visualization takes the form of a conditional distribution: it shows how the measured outcomes deviate when tampering occurs compared to when the channel is secure. By highlighting the difference between expected correlations and disturbed ones, the visualization reinforces the principle that tampering cannot occur undetected in entangled systems. For undergraduates, this connects the theory of quantum correlations with the practical goal of tamper-evident communication.

*Book Reference: Chapter 10 — Quantum Communication for Distributed AI Systems*

This exercise shows how entanglement can be used to detect tampering or interference. It parallels the chapter’s discussion of secure multi-node learning and integrity verification in distributed QAIS pipelines.


**Expected Results**

The untampered results should only include 00 and 11. After tampering, additional outcomes (01, 10) should appear, clearly showing that interference has occurred.

Untampered: Outcomes concentrate on 00 and 11 (same-bit parity), reflecting the correlations of ∣Φ⁺⟩.

Tampered (X on qubit 1): Outcomes shift to 01 and 10 (opposite-bit parity), consistent with transforming ∣Φ⁺⟩ → ∣Ψ⁺⟩.

The clear diagonal ↔ anti-diagonal swap is a tamper signature. On noisy backends, small leakage to the other bins may appear, but the parity preference remains visible.

In [ ]:
# Lab 10 — Entanglement-Assisted Tamper Check

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

# 1) Build Bell circuit |Φ+> = (|00> + |11>)/√2
def bell_measure_circuit(tamper: bool = False) -> QuantumCircuit:
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    # Simple disturbance model: flip one qubit (change correlation parity)
    if tamper:
        qc.x(1)  # try qc.z(1) or calibrated noise models in advanced labs
    qc.measure([0, 1], [0, 1])
    return qc

# 2) Simulate both cases
shots = 4096
backend = Aer.get_backend("aer_simulator")

qc_clean  = bell_measure_circuit(tamper=False)
qc_tamper = bell_measure_circuit(tamper=True)

res_clean  = backend.run(transpile(qc_clean, backend),  shots=shots, seed_simulator=42).result()
res_tamper = backend.run(transpile(qc_tamper, backend), shots=shots, seed_simulator=43).result()

counts_clean  = res_clean.get_counts()
counts_tamper = res_tamper.get_counts()

# 3) Plot distributions
fig = plot_histogram([counts_clean, counts_tamper],
                     legend=["Untampered", "Tampered"],
                     title="Tampered vs Untampered Distributions")
ax = fig.axes[0]
ax.set_xlabel("Bitstring")
ax.set_ylabel("Counts")

# 4) Save with correct IEEE label (Part 1 => Beginner_Labs/figures)
save_e_figure("Figure E.1.10", "P1_Lab10_Tamper_Check_Hist.png",
              subdir="Beginner_Labs/figures", fig=fig, ax=ax)

plt.show()
plt.close(fig)


Figure E.1.10 — Entanglement-Assisted Tamper Check.
The histograms compare measurement outcomes for an entangled Bell pair with and without a deliberate disturbance. Untampered runs cluster on 00 and 11, indicating same-bit correlations; tampered runs shift weight to 01 and 10, indicating opposite-bit correlations. The parity change reveals the disturbance.

**Methodology Analysis**

Prepare the Bell state ∣Φ⁺⟩ = (∣00⟩ + ∣11⟩)/√2, measure both qubits in the computational basis, and record outcome counts. Repeat after introducing a simple tamper operation (here, X on qubit 1) that flips one qubit post-entanglement. Plot both distributions side-by-side to visually detect a parity change in the correlations.

**Technical Analysis (for the Visual)**

An X on the second qubit maps
∣Φ⁺⟩ = (∣00⟩ + ∣11⟩)/√2 → (∣01⟩ + ∣10⟩)/√2 = ∣Ψ⁺⟩,
flipping the ZZ correlation from +1 (same bits favored) to −1 (opposite bits favored). In the histogram, that manifests as a move from 00/11 to 01/10 dominance. This is a minimal instance of entanglement-assisted integrity checking: you authenticate the correlation pattern, not the individual bits. Any process that changes the correlation parity (bit flips, certain phase flips plus basis changes, or decoherence) is detectable.

**Intuition Sidebar**

Like a fair coin, the qubit is balanced between two outcomes until measured.

Think of two synchronized coins that always match. If someone secretly flips one coin after they synchronize, the pair suddenly prefers opposite faces. You might not see the flip itself, but you do see the parity change. That parity flip is your tamper alarm.

**Conclusion — Lab 10**

This lab shows how entanglement correlations act as a tamper-evident seal. A simple local disturbance converts ∣Φ⁺⟩ correlations into ∣Ψ⁺⟩ correlations, producing a visible signature in the measurement histogram. In quantum AI systems and quantum networks, monitoring correlation parity provides a lightweight integrity check that complements more sophisticated verification or QKD protocols.

**Key Takeaways**

Entanglement enables correlation-based integrity checks; you verify parity, not payload bits.

A local disturbance can invert correlation parity (00/11 ↔ 01/10), which is easy to detect in simple measurements.

Even with noise, parity trends persist, making this approach practical for near-term devices.

**Congratulations**

Congratulations on completing Lab 10 — Entanglement Tamper Check! You implemented and analyzed a tamper-evident mechanism that leverages entanglement’s unique correlation structure. This hands-on capability maps directly to quantum integrity monitoring and security-aware quantum AI workflows, the kinds of techniques used by practitioners building reliable quantum systems today.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 10 — Quantum Communication for Distributed AI Systems**:  
- Questions 3 and 4 (entanglement verification and tamper detection).  
These reinforce the QBER and distribution-shift concepts demonstrated in **E.2 Lab 5** (Figure E.2.5).

---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.


---